# License

In [ ]:
# ------------------------------------------------------------------------------
#  QuantumThermal sample, written in Python
#
#  Multi-Scale Modeling Laboratory (SMaLL)
#  Website: https://small.polito.it/
#  GitHub repository: https://github.com/SMaLL-PoliTo/QuantumThermal
#
#  Copyright (C) 2025 Pietro Asinari, Matteo Maria Piredda,
#  Giulio Barletta, Paolo De Angelis
#  E-mail contact: pietro.asinari@polito.it
#
#  This code is licensed under the MIT License.
#  You may obtain a copy of the License at
#
#      https://opensource.org/licenses/MIT
#
#  Permission is hereby granted, free of charge, to any person obtaining a copy
#  of this software and associated documentation files (the "Software"), to deal
#  in the Software without restriction, including without limitation the rights
#  to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
#  copies of the Software, and to permit persons to whom the Software is
#  furnished to do so, subject to the following conditions:
#
#  The above copyright notice and this permission notice shall be included in
#  all copies or substantial portions of the Software.
#
#  THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
#  IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
#  FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
#  AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
#  LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
#  OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
#  SOFTWARE.
# ------------------------------------------------------------------------------

# Solving the Heat Equation with Quantum Computing

This notebook demonstrates a minimal pipeline to advance one time step of the 1D heat equation via a classical implicit finite-difference (FD) scheme, and then map the resulting linear system to a quantum expectation-minimization problem solvable with a Variational Quantum Eigensolver (VQE). The goal is to compare the FD reference with a simulated quantum solution under ideal (statevector) conditions.

**Workflow**
1. Build the FD system $A\,x=b$ for one time step.
2. Normalize $(A,b)$ to be compatible with quantum state amplitudes.
3. Define an observable (Hamiltonian) whose ground state encodes the target solution.
4. Specify an ansatz and cost function, and run VQE (Qiskit primitives).
5. Post-process and de-normalize to recover physical temperature fields.

**Environment**: **`QTeaLeaves`**.

# Preparation

Before running the code, it is necessary to create a conda environment as following:
`conda create -n thermal-science -y python=3.11 ipykernel`

Then activate the environment:
`conda activate thermal-science`

Required packages will be installed with the first line of the code

In [1]:
%pip install numpy scipy matplotlib cvxpy pylatexenc
%pip install qtealeaves
%pip install qredtea 
%pip install qmatchatea
%pip install qiskit==1.4.2 qiskit-aer==0.17.2 qiskit-algorithms==0.3.1
%pip install qrisp==0.7.9

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import inv, eigvalsh
from scipy.optimize import minimize
import statistics
import time

from qiskit.circuit.library import TwoLocal
from qiskit.primitives import (
    StatevectorEstimator as Estimator,
    StatevectorSampler as Sampler,
)
from qiskit.quantum_info import SparsePauliOp, Statevector

from qmatchatea import QCOperators, run_simulation
import qtealeaves.observables as obs

In [3]:
from IPython.display import clear_output

clear_output()

## Finite-Difference Scheme for Heat Conduction
We model the heat diffusion on a 1D spatial mesh with periodic boundary conditions using an implicit finite difference scheme:

- The temperature distribution at time $t=0$, $T_{old}$, is initialized using a sinusoidal perturbation.
- A matrix C is constructed based on the Fourier number r to represent the linear system.
- The new temperature distribution T_new is obtained by solving $C\cdot T_{new} = T_{old}$.

Visualization shows $T_{old}$ vs $T_{new}$.

In [ ]:
n = 3  # number of qubits, num_qubits
N = pow(2, n)  # number of mesh nodes

# T_old
T_old = np.ones(N)
for i in range(N):
    T_old[i] = 1 + (1 / 2) * np.sin(2 * np.pi * (i + 1) / N)
print(T_old)

# Linear system C*T_new = T_old
# T_new = inv(C)*T_old
r = 0.5  # = delta_t*alpha/(delta_x**2) = Fo, Fourier number
d = np.ones(N) * (1 + 2 * r)
od = np.ones(N - 1) * (-r)
C = np.diag(d, 0) + np.diag(od, -1) + np.diag(od, 1)
C[0, -1] = C[-1, 0] = -r
print(C)

# T_new
T_new = inv(C) @ T_old
print(T_new)

In [ ]:
plt.plot(T_old, label="T_old (t = 0)")
plt.plot(T_new, label=r"T_new (t = $\Delta$t)", linestyle="dashed")
plt.legend()
plt.xlabel("spatial mesh")
plt.ylabel("temperature")
plt.show()

## Normalization for Quantum Mapping
We normalize both $T_{old}$ and $T_{new}$ for compatibility with quantum state amplitudes (unit norm vectors):

1. Compute norms $b_0$ and $x_0$.
2. Define normalized vectors $b$ and $x_{th}$.
3. Construct normalized linear system $A \cdot x_{th} = b$, which will be used in quantum optimization.

In [ ]:
# Normalized linear system A*x_th = b

# T_old
TT_old = np.sum(T_old**2)
b0 = np.sqrt(TT_old)
print(b0)
b = T_old / b0

# T_new
TT_new = np.sum(T_new**2)
x0 = np.sqrt(TT_new)
x_th = T_new / x0  # (theoretical)

# Linear system (normalized)
f = x0 / b0
A = C * f  # (normalized)

In [ ]:
plt.plot(b, label="b")
plt.plot(x_th, label="x_th", linestyle="dashed")
plt.legend()
plt.xlabel("spatial mesh")
plt.ylabel("normalized temperature")
plt.show()

# Constructing the Quantum Observable
We define the observable operator for the VQE to minimize:

$O=A^\dagger\left(I-| b \rangle\langle b |\right)A$

This is the quantum observable for $T_{new}$, or better for $| x \rangle$ (normalized), such that:

$\langle x | \, O \, | x \rangle \xrightarrow{} 0$ (minimum) 

because $A^\dagger = f C^\dagger$, and then
$O=f^2 C^\dagger \left(I-| b \rangle\langle b |\right) C$
which has the same minimum for $| x \rangle$.

Ref: PHYSICAL REVIEW A 107, 052422 (2023)
Depth analysis of variational quantum algorithms for the heat equation

In [9]:
O = np.eye(N) - np.outer(b, b)
O = C.T @ O @ C

In [ ]:
observable = SparsePauliOp.from_operator(O)
print(observable)
print(len(observable.paulis))

In [ ]:
eigenvalues = eigvalsh(O)
print(min(eigenvalues))

# Variational Quantum Eigensolver (VQE)
We use `Qiskit`'s estimator primitive to compute the expectation value of our observable under a parameterized ansatz.

## Loss/cost function for VQE
A custom cost function is defined for quantum optimization using `Qiskit`'s Estimator.
It returns the expectation value of the observable for a given ansatz and parameter set.

In [15]:
def cost_func_vqe(params, ansatz, hamiltonian, estimator):
    """Return estimate of energy from estimator

    Parameters:
        params (ndarray): Array of ansatz parameters
        ansatz (QuantumCircuit): Parameterized ansatz circuit
        hamiltonian (SparsePauliOp): Operator representation of Hamiltonian
        estimator (Estimator): Estimator primitive instance

    Returns:
        float: Energy estimate
    """
    pub = (ansatz, hamiltonian, params)
    cost = estimator.run([pub]).result()[0].data.evs

    return cost

## Ansatz
We define the parameterized quantum circuit (ansatz) using a custom `TwoLocal` ansatz.

The number of parameters and circuit structure are displayed.

In [ ]:
num_qubits = n

# Best rational ansatz with minimum number of parameters
raw_ansatz = TwoLocal(num_qubits, rotation_blocks="ry", entanglement_blocks="cz")

raw_ansatz.decompose().draw("mpl")

In [ ]:
raw_ansatz.num_parameters

## Quantum Simulation Setup
We instantiate `StatevectorEstimator` and `StatevectorSampler` from Qiskit.

Initial parameter values $\theta_0$ are set to ones.

In [16]:
estimator = Estimator()
sampler = Sampler()

# Initial (arbitrary) set of parameter
theta0 = np.ones(raw_ansatz.num_parameters)

## Quantum minimization / VQE optimization

We minimize the cost function using `SciPy`'s minimize with the `COBYLA` algorithm:

Optimizes ansatz parameters to minimize $\langle x | \, O \, | x \rangle$.

The result gives optimized parameters and circuit.

We visualize the output probability distribution and extract amplitudes as the quantum solution x.

See [here](https://learning.quantum.ibm.com/tutorial/variational-quantum-eigensolver) for more details.

In [ ]:
start_time = time.time()

result = minimize(
    cost_func_vqe,
    theta0,
    args=(raw_ansatz.decompose(), observable, estimator),
    method="COBYLA",
    tol=1e-3,  # better tolerance is possible but it would require more iterations/time
    options={"maxiter": 10000, "disp": True},
)
print(result)

end_time = time.time()
execution_time = end_time - start_time
print(f"Execution Time (s): {execution_time}")

In [ ]:
theta = result.x  # Final optimized parameters
ansatz = raw_ansatz.assign_parameters(theta)
ansatz.decompose().draw("mpl")

## Simulated quantum results

In [19]:
ideal_distribution = Statevector.from_instruction(ansatz).probabilities_dict()

In [20]:
# Solution (quantum)
y = np.real(list(ideal_distribution.values()))  # probabilities
x = np.sqrt(y)  # amplitudes

In [ ]:
plt.plot(b, label="b")
plt.plot(x_th, label="x_th (FD)", linestyle="dashed")
plt.plot(x, "bo", label="x (qiskit)")
plt.legend()
plt.xlabel("spatial mesh")
plt.ylabel("normalized temperature")
plt.show()

## Post-processing & De-normalization
Recover temperature in original units from normalized quantum solution using mean-scaling.

Compare classical $T_{new}$, quantum $T_{new,q}$, and original $T_{old}$.

In [ ]:
x_mean = statistics.mean(x)
T_mean = statistics.mean(T_old)
scale = T_mean / x_mean
print(f"\nscale = {scale}")

T_new_q = x * scale

In [ ]:
plt.plot(T_old, label="T_old (t = 0)")
plt.plot(T_new, label=r"T_new (FD) (t = $\Delta$t)", linestyle="dashed")
plt.plot(T_new_q, "bo", label=r"T_new_q (qiskit) (t = $\Delta$t)")
plt.legend()
plt.xlabel("spatial mesh")
plt.ylabel("temperature")
plt.show()

# Simulated quantum results (quantum tea)

## Observable conversion: `SparsePauliOp` → `qtealeaves`

We convert Qiskit’s `SparsePauliOp` into the dictionary/list format expected by `qtealeaves`.

`sparse_pauli_to_qtealeaves_dict`:
- iterates each term of the `SparsePauliOp`
- translates Pauli strings (e.g., XIZ) into per-qubit operators in Quantum Tea’s format,
- preserves real coefficients; tiny imaginary parts from numerics are dropped.

Conventions / gotchas:
- Qubit order: the same convention as the ansatz/circuit construction (Qiskit endianness). Keep this consistent across frameworks.
- Constant term: global energy shift is kept as a separate constant where applicable.

PSD check: if the observable is used as a loss, ensure it is positive semi-definite (or add a shift) so that the ground state corresponds to the target solution.

In [27]:
def sparse_pauli_to_qtealeaves_dict(sparse_op):
    paulis_list = []
    for pauli, coeff in zip(sparse_op.paulis, sparse_op.coeffs):
        paulis_list.append(
            {
                "label": pauli.to_label(),
                "coeff": {"real": float(coeff.real), "imag": float(coeff.imag)},
            }
        )
    return {"paulis": paulis_list}

In [25]:
pauli_dict_hamiltonian = sparse_pauli_to_qtealeaves_dict(observable)

obsv = obs.TNObservables()
ham = obs.TNObsWeightedSum.from_pauli_string("hamiltonian", pauli_dict_hamiltonian)
obsv += ham

We define `vqe_step` to bind parameters θ to the ansatz and asks Quantum Tea to evaluate

$E(\theta) = \langle \psi (\theta) | O | \psi (\theta) \rangle$

where $O$ is the converted observable, and to return a scalar energy used by the classical optimizer.

In [28]:
def vqe_step(theta, ansatz, observables):
    """
    Single step for the vqe: run the circuit
    with a given set of parameters and measure
    the observables.

    Parameters
    ----------
    theta : list
    ansatz : QuantumCircuit
        parametric qiskit quantum circuit
    observables : :class:`TNObservables`
        Observables to measure. There should be an obervable named Hamiltonian

    Returns
    -------
    float
        Expectation value of the hamiltonian
    """

    ansatz.decompose().draw("mpl")

    # Define the Z, X operators
    ops = QCOperators()
    ops["X"] = np.array([[0, 1], [1, 0]])
    ops["Z"] = np.array([[1, 0], [0, -1]])
    # bind the parametric ansatz to the parameters
    binded_ansatz = ansatz.assign_parameters(theta).decompose()
    # Run the simulation
    res = run_simulation(binded_ansatz, observables=observables, operators=ops)

    return np.real(res.observables["hamiltonian"])

In [ ]:
def vqe_step_opt(x):
    return vqe_step(x, raw_ansatz, obsv)

initial_guess = np.random.normal(np.pi / 2, 0.05, raw_ansatz.num_parameters)

initial_energy = vqe_step_opt(initial_guess)
print(f"Initial energy: {initial_energy}")

res = minimize(vqe_step_opt, initial_guess, method="BFGS")

final_energy = vqe_step_opt(res.x)
print(f"Final energy: {final_energy}, expected {0:.5f}")

In [ ]:
theta = res.x  # Final optimized parameters
ansatz = raw_ansatz.assign_parameters(theta)
ansatz_reversed = ansatz.reverse_bits()
ansatz_reversed.decompose().draw("mpl")

In [ ]:
ideal_distribution = Statevector.from_instruction(ansatz_reversed).probabilities_dict()

# Solution (quantum)
y = np.real(list(ideal_distribution.values()))  # probabilities
x = np.sqrt(y)  # amplitudes

x_mean = statistics.mean(x)
T_mean = statistics.mean(T_old)
scale = T_mean / x_mean
T_new_q_tea = x * scale

plt.plot(T_old, label="T_old (t = 0)")
plt.plot(T_new, label=r"T_new (FD) (t = $\Delta$t)", linestyle="dashed")
plt.plot(T_new_q, "bo", label=r"T_new_q (qiskit) (t = $\Delta$t)")
plt.plot(
    T_new_q_tea,
    color="skyblue",
    marker="+",
    label=r"T_new_q (quantum tea) (t = $\Delta$t)",
)
plt.legend()
plt.xlabel("spatial mesh")
plt.ylabel("temperature")
plt.show()